# Multiclass (MEL vs BCC vs SCC): reducción de dimensionalidad con PCA y LDA

Parte de `03_v5_ml_multiclass_rus_smote.ipynb` (mismo remuestreo: **RUS + SMOTE hasta la clase del medio**, mismos
modelos y métricas) y lo único que cambia es **cómo se reducen las 412 features** antes del modelo. Se comparan en la
misma CV:

| Variante | Qué hace | ¿Supervisada? |
|---|---|---|
| `todas` | las 412 features | — |
| `top-K` | las K features más importantes según un Random Forest | sí |
| `PCA` | proyecta en las direcciones de **máxima varianza** (componentes = mezclas de features) | no |
| `LDA` | proyecta en las direcciones que **mejor separan las clases** (con 3 clases, máximo 2) | sí |

1. Carga de datos y **visualización con PCA y LDA**: varianza explicada, scatter 2D / 3D / matriz de los primeros
   componentes, loadings (qué features y familias pesan en cada componente) y proyección LDA de train y val.
2. CV estratificada de 5 folds en train con **todas las variantes de `REDUCTIONS` × todos los modelos**, más un resumen
   en heatmaps (kappa, balanced accuracy y recall de SCC).
3. Train vs CV y ablación por familias.
4. Evaluación final en **val** con la variante y el modelo que mejor salen en la CV, más una tabla **informativa** con el
   mejor modelo de cada variante en val (para ver si val confirma la CV, **no** para elegir).
5. Importancia por permutación y control de sesgo de adquisición.

**Dónde va cada paso (todo se ajusta solo con el train de cada fold):**

```
todas / top-K :  imputación → [top-K] → cuantil → RUS+SMOTE → modelo
PCA / LDA     :  imputación → cuantil → StandardScaler → PCA / LDA → RUS+SMOTE → modelo
```

- **El escalado tiene que ir antes de PCA/LDA.** PCA busca varianza; sin escalar, mandarían las features con valores
  más grandes por su unidad, no por su información. El cuantil (→ normal) ya deja casi todas las features con media 0 y
  desviación 1, y el `StandardScaler` lo garantiza exactamente, también en las features con muchos valores repetidos.
- PCA y LDA se ajustan **antes** del remuestreo, con los datos reales, igual que el top-K. Así SMOTE interpola en el
  espacio reducido. LDA usa *priors* iguales para que la proyección no la dicte el desbalance.
- PCA **no** usa las clases, así que puede conservar varianza que no discrimina. Suele ayudar a LogReg, SVM y KNN y
  perjudicar a los árboles (cortan eje a eje y los componentes mezclan todas las features). LDA sí usa las clases,
  pero se queda en 2 dimensiones.
- **El nº de componentes es un hiperparámetro:** se prueban varios en la CV (`REDUCTIONS`). Como referencia,
  la sección 1 imprime cuántos hacen falta para el 80/90/95/99 % de la varianza. `("pca", 0.95)` usa el nº que da el 95 %
  **en el train de cada fold**.
- Se añade **LDA como clasificador** a la lista de modelos (lineal, rápido, y natural junto a la proyección LDA).
- Cada variante de `REDUCTIONS` es una CV completa: quita las que no te interesen si tarda demasiado.

**Métricas:** la principal es **Cohen's kappa**; también balanced accuracy, F1 macro, AUC ROC OvR macro, AP macro y el
**recall de cada clase** (sobre todo SCC). Ojo: el kappa da el mismo peso a cada **imagen**, no a cada clase, así que
apenas refleja SCC; para eso mira `bal_acc` y `recall_scc`.

In [ ]:
import os
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, average_precision_score, balanced_accuracy_score,
                             classification_report, cohen_kappa_score, f1_score, precision_recall_curve,
                             make_scorer, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import QuantileTransformer, StandardScaler, label_binarize
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight

try:
    from imblearn import FunctionSampler
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.pipeline import Pipeline as ImbPipeline     # aplica el sampler solo al entrenar (fit), nunca al predecir
except ImportError as e:
    raise ImportError("Falta imbalanced-learn: pip install imbalanced-learn (en el entorno del kernel)") from e

try:
    from xgboost import XGBClassifier
except Exception as e:      # ImportError, o versiones mezcladas de conda/pip (ValueError)
    import sys
    XGBClassifier = None
    print(f"AVISO: no se puede importar xgboost -> se omite XGBoost\n"
          f"  error: {type(e).__name__}: {e}\n  python del kernel: {sys.executable}")

warnings.filterwarnings("ignore")

# ---------------- Configuración ----------------
# busca la raíz del proyecto (la carpeta que contiene features_multiclass/) desde la carpeta del notebook hacia arriba
# ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "features_multiclass").exists()), Path.cwd())
ROOT = Path(r"C:\Users\jcgon\PycharmProjects\Melanoma-Challenge-ML-UdG")    # o fíjala a mano
assert (ROOT / "features_multiclass").exists(), f"ROOT incorrecta: {ROOT} (fíjala a mano arriba)"

TRAIN_CSV = ROOT / "features_multiclass" / "features_train.csv"
VAL_CSV   = ROOT / "features_multiclass" / "features_val.csv"
OUT_DIR   = ROOT / "ml" / "ml_results_multiclass_pca_lda"      # carpeta propia: no pisa los resultados de los otros notebooks
CLASSES = ["bcc", "mel", "scc"]          # orden de las etiquetas 0, 1, 2
# Variantes de reducción de dimensionalidad que se comparan en la CV (todas con el mismo remuestreo).
#   None         -> todas las features
#   ("topk", K)  -> top-K features según un Random Forest
#   ("pca", n)   -> PCA: n = nº de componentes (int) o fracción de varianza a conservar (float, p. ej. 0.95)
#   ("lda", 2)   -> LDA (supervisada): con 3 clases, como mucho 2 componentes
# Cada una es una CV completa (todos los modelos x 5 folds): quita las que no quieras si tarda mucho.
REDUCTIONS = {
    "todas": None,
    "top-30": ("topk", 30),
    "top-50": ("topk", 50),
    "top-100": ("topk", 100),
    "PCA 95%": ("pca", 0.95),
    "PCA-10": ("pca", 10),
    "PCA-30": ("pca", 30),
    "PCA-60": ("pca", 60),
    "LDA-2": ("lda", 2),
}
CV_FOLDS = 5
SEED = 42
N_JOBS = max(1, (os.cpu_count() or 1) - 2)          # todos los cores disponibles menos 2

TARGET = "middle"          # "middle" = nº de imágenes de la clase del medio (en el train de cada fold), o un número fijo (p. ej. 1500)
SMOTE_K = 5                # vecinos de SMOTE


def rus_smote_to(X, y, target="middle", k_neighbors=5, seed=42):
    """Clases por encima de `target` -> random undersampling hasta `target`;
    clases por debajo -> SMOTE hasta `target`. Las que ya tienen `target` no se tocan."""
    counts = pd.Series(y).value_counts()
    n = int(np.median(counts)) if target == "middle" else int(target)
    down = {c: n for c, m in counts.items() if m > n}
    up = {c: n for c, m in counts.items() if m < n}
    if down:
        X, y = RandomUnderSampler(sampling_strategy=down, random_state=seed).fit_resample(X, y)
    if up:
        X, y = SMOTE(sampling_strategy=up, k_neighbors=k_neighbors, random_state=seed).fit_resample(X, y)
    return X, y


RESAMPLING = f"RUS+SMOTE ({TARGET})"
# FunctionSampler lo convierte en un sampler normal de imblearn: va dentro del ImbPipeline y solo actúa en fit()
SAMPLER = FunctionSampler(func=rus_smote_to, kw_args={"target": TARGET, "k_neighbors": SMOTE_K, "seed": SEED})
SCALE_ALL = True                                   # True: transformación cuantil en todos los modelos, antes del sampler
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"ROOT: {ROOT}\ncores: {os.cpu_count()} -> N_JOBS = {N_JOBS}\nremuestreo: {RESAMPLING}")

## 1. Carga de datos

In [ ]:
tr_all, va_all = pd.read_csv(TRAIN_CSV), pd.read_csv(VAL_CSV)
tr = tr_all[tr_all["subtype"].isin(CLASSES)].reset_index(drop=True)
va = va_all[va_all["subtype"].isin(CLASSES)].reset_index(drop=True)

PREFIXES = ("shape_", "color_", "tex_", "gcolor_", "gtex_")
FEATS = [c for c in tr.columns if c.startswith(PREFIXES)]
assert FEATS == [c for c in va.columns if c.startswith(PREFIXES)], "train y val tienen columnas distintas"

CLS2ID = {c: i for i, c in enumerate(CLASSES)}
Xtr, ytr = tr[FEATS].replace([np.inf, -np.inf], np.nan), tr["subtype"].map(CLS2ID).to_numpy()
Xva, yva = va[FEATS].replace([np.inf, -np.inf], np.nan), va["subtype"].map(CLS2ID).to_numpy()
family = lambda c: c.split("_")[0]

counts = pd.DataFrame({"train": tr["subtype"].value_counts(), "val": va["subtype"].value_counts()}).reindex(CLASSES)
counts["train_%"] = (100 * counts["train"] / counts["train"].sum()).round(1)
counts["val_%"] = (100 * counts["val"] / counts["val"].sum()).round(1)
print(counts.to_string())
print(f"\ntrain: {Xtr.shape} | val: {Xva.shape} | NaN train {int(Xtr.isna().sum().sum())}, val {int(Xva.isna().sum().sum())}")
print("features por familia:", pd.Series([family(c) for c in FEATS]).value_counts().to_dict())

skf = StratifiedKFold(CV_FOLDS, shuffle=True, random_state=SEED)
FAM_COLOR = {"shape": "#1f77b4", "color": "#d62728", "tex": "#2ca02c", "gcolor": "#ff9896", "gtex": "#98df8a"}

### Número de imágenes de cada clase
En train, en val y en cada fold de la CV (la división estratificada mantiene la proporción). También cuántas imágenes
de cada clase ve un modelo **después del remuestreo**, con el train del fold 1 como ejemplo. El remuestreo se rehace
en cada fold y en el entrenamiento final, siempre solo sobre train.

In [ ]:
n_tr = tr["subtype"].value_counts().reindex(CLASSES)
n_va = va["subtype"].value_counts().reindex(CLASSES)

# lo que ve cada modelo tras el remuestreo, con el train del fold 1 de ejemplo (val no se toca nunca)
id2cls = dict(enumerate(CLASSES))
a0, _ = next(skf.split(np.zeros(len(ytr)), ytr))
Xa0 = SimpleImputer(strategy="median").fit_transform(Xtr.iloc[a0])
_, ya0_res = clone(SAMPLER).fit_resample(Xa0, ytr[a0])
n_f1 = pd.Series(ytr[a0]).map(id2cls).value_counts().reindex(CLASSES)
n_res = pd.Series(ya0_res).map(id2cls).value_counts().reindex(CLASSES)

summary = pd.DataFrame({"train": n_tr, "train_%": (100 * n_tr / n_tr.sum()).round(1),
                        "val": n_va, "val_%": (100 * n_va / n_va.sum()).round(1),
                        "fold1 train": n_f1, f"fold1 train tras {RESAMPLING}": n_res,
                        "factor": (n_res / n_f1).round(2)})
summary.loc["TOTAL"] = [n_tr.sum(), 100.0, n_va.sum(), 100.0, n_f1.sum(), n_res.sum(), round(n_res.sum() / n_f1.sum(), 2)]
print(summary.to_string(), "\n")

per_fold = pd.DataFrame({f"fold {f + 1} ({part})": pd.Series(ytr[idx]).map(id2cls).value_counts().reindex(CLASSES)
                         for f, (a, b) in enumerate(skf.split(np.zeros(len(ytr)), ytr))
                         for part, idx in (("train", a), ("val", b))})
print("Imágenes por clase en cada fold de la CV (antes de remuestrear):")
print(per_fold.to_string())

### Comprobación: ¿son las mismas imágenes que las del challenge multiclass?
Si existe `Multiclass/<train|val>/<clase>/`, compara los nombres de archivo con los de las features.

In [ ]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
mc_dir = ROOT / "Multiclass"
checked = False
for split, df_ in (("train", tr), ("val", va)):
    d = next((p for p in mc_dir.iterdir() if p.is_dir() and p.name.lower().startswith(split)), None) if mc_dir.exists() else None
    if d is None:
        continue
    checked = True
    names = {f.stem for f in d.rglob("*") if f.suffix.lower() in IMG_EXT}
    ours = set(df_["name"])
    print(f"{split}: imágenes en Multiclass {len(names)} | con features {len(ours)} | comunes {len(names & ours)} | "
          f"solo en Multiclass {len(names - ours)} | solo en features {len(ours - names)}")
if not checked:
    print("No hay carpeta Multiclass/<train|val>/ con imágenes: no se puede comprobar.\n"
          "Los tamaños sí coinciden con el enunciado si train ≈ 5082 y val ≈ 1270.")

## 1b. Visualización con PCA y LDA
Solo para **explorar**: aquí PCA y LDA se ajustan con **todo train** (la CV de la sección 2 los ajusta dentro de cada
fold). Mismo preprocesado que en el pipeline: imputación → cuantil → `StandardScaler`.

1. **Varianza explicada acumulada**: cuántos componentes hacen falta para el 80/90/95/99 % (referencia para elegir el nº).
2. **Scatter** de los primeros componentes: PC1-PC2, PC1-PC2-PC3 en 3D y una matriz con los `N_PLOT` primeros.
   Es normal que las clases salgan muy solapadas: 2-3 componentes de 412 recogen poca varianza, y la varianza no tiene
   por qué ser lo que separa las clases. Que se solapen aquí **no** quiere decir que el modelo no las separe.
3. **Loadings** (explicabilidad): qué features pesan más en cada componente y cuánto pesa cada **familia**.
4. **LDA**: proyección en las 2 direcciones que mejor separan las clases, en train (optimista, es con lo que se ajusta)
   y en **val** (honesto).

In [ ]:
CLS_COLOR = {"bcc": "#ff7f0e", "mel": "#1f77b4", "scc": "#2ca02c"}
N_PLOT = 4             # nº de componentes de PCA en la matriz de scatter (2, 3, 4...)
N_LOAD = 3             # nº de componentes de los que se enseñan los loadings

viz_prep = Pipeline([("imp", SimpleImputer(strategy="median"))] + [
    ("scale", QuantileTransformer(n_quantiles=200, output_distribution="normal", random_state=SEED)),
    ("std", StandardScaler())]).fit(Xtr)
Ztr, Zva = viz_prep.transform(Xtr), viz_prep.transform(Xva)

pca_viz = PCA(random_state=SEED).fit(Ztr)
evr = pca_viz.explained_variance_ratio_
cum = np.cumsum(evr)
n_for = {t: int(np.searchsorted(cum, t) + 1) for t in (0.80, 0.90, 0.95, 0.99)}
print("componentes necesarios para explicar:", {f"{int(t * 100)}%": n for t, n in n_for.items()}, f"(de {len(FEATS)})")
print("varianza explicada por los primeros:", {f"PC{i + 1}": f"{100 * v:.1f}%" for i, v in enumerate(evr[:6])})

fig, axs = plt.subplots(1, 2, figsize=(13, 4))
axs[0].plot(np.arange(1, len(cum) + 1), cum)
for t, n in n_for.items():
    axs[0].axhline(t, ls=":", lw=0.8, color="gray")
    axs[0].axvline(n, ls=":", lw=0.8, color="gray")
    axs[0].annotate(f"{int(t * 100)}%: {n}", (n, t), textcoords="offset points", xytext=(4, -12), fontsize=8)
axs[0].set(xscale="log", xlabel="nº de componentes (escala log)", ylabel="varianza explicada acumulada",
           title="PCA en train: varianza acumulada")
axs[1].bar(np.arange(1, 31), 100 * evr[:30])
axs[1].set(xlabel="componente", ylabel="% de varianza", title="Varianza de cada componente (los 30 primeros)")
plt.tight_layout(); plt.show()

In [ ]:
P = pca_viz.transform(Ztr)
style = lambda c: dict(s=10 if c == "scc" else 3, alpha=0.8 if c == "scc" else 0.3, color=CLS_COLOR[c])   # SCC encima

fig = plt.figure(figsize=(14, 6))
ax = fig.add_subplot(1, 2, 1)
for j, c in enumerate(CLASSES):
    m = ytr == j
    ax.scatter(P[m, 0], P[m, 1], label=f"{c} ({m.sum()})", **style(c))
ax.set(xlabel=f"PC1 ({100 * evr[0]:.1f}%)", ylabel=f"PC2 ({100 * evr[1]:.1f}%)", title="PCA (train): PC1 vs PC2")
ax.legend(markerscale=3)
ax = fig.add_subplot(1, 2, 2, projection="3d")
for j, c in enumerate(CLASSES):
    m = ytr == j
    ax.scatter(P[m, 0], P[m, 1], P[m, 2], label=c, **style(c))
ax.set(xlabel="PC1", ylabel="PC2", zlabel="PC3", title=f"PC1-PC2-PC3 ({100 * cum[2]:.1f}% de la varianza)")
plt.tight_layout(); plt.show()

# matriz de scatter de los N_PLOT primeros componentes (diagonal: distribución de cada clase en ese componente)
fig, axs = plt.subplots(N_PLOT, N_PLOT, figsize=(2.7 * N_PLOT, 2.7 * N_PLOT))
for i in range(N_PLOT):
    for j in range(N_PLOT):
        ax = axs[i, j]
        for k, c in enumerate(CLASSES):
            m = ytr == k
            if i == j:
                ax.hist(P[m, i], bins=40, density=True, histtype="step", color=CLS_COLOR[c], label=c)
            else:
                ax.scatter(P[m, j], P[m, i], **{**style(c), "s": 2})
        if i == N_PLOT - 1:
            ax.set_xlabel(f"PC{j + 1}")
        if j == 0:
            ax.set_ylabel(f"PC{i + 1}")
        ax.tick_params(labelsize=6)
axs[0, 0].legend(fontsize=7)
plt.suptitle(f"Los {N_PLOT} primeros componentes de PCA (train)"); plt.tight_layout(); plt.show()

In [ ]:
# Loadings: peso de cada feature original en cada componente (las features están estandarizadas, así que son comparables)
load = pd.DataFrame(pca_viz.components_[:max(N_LOAD, 10)].T, index=FEATS,
                    columns=[f"PC{i + 1}" for i in range(max(N_LOAD, 10))])

fig, axs = plt.subplots(1, N_LOAD, figsize=(5.2 * N_LOAD, 5.5))
for ax, pc in zip(np.atleast_1d(axs), load.columns[:N_LOAD]):
    top = load[pc].abs().sort_values(ascending=False).head(15).index[::-1]
    ax.barh(top, load.loc[top, pc], color=[FAM_COLOR[family(c)] for c in top])
    ax.axvline(0, color="k", lw=0.6)
    ax.set_title(f"{pc} ({100 * evr[int(pc[2:]) - 1]:.1f}% var): top-15 loadings")
    ax.tick_params(axis="y", labelsize=7)
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in FAM_COLOR.values()]
fig.legend(handles, FAM_COLOR.keys(), loc="lower center", ncol=len(FAM_COLOR))
plt.tight_layout(rect=(0, 0.05, 1, 1)); plt.show()

# Peso de cada familia en cada componente: suma de los loadings al cuadrado (cada componente suma 1).
# Compáralo con la proporción de features de cada familia: si una familia pesa mucho más que su proporción, ese
# componente es "sobre todo" esa familia.
fam_w = (load ** 2).groupby(lambda c: family(c)).sum().reindex(list(FAM_COLOR))
fam_w["% de features"] = pd.Series([family(c) for c in FEATS]).value_counts(normalize=True).reindex(list(FAM_COLOR))
print("Peso de cada familia en los 10 primeros componentes:\n")
print(fam_w.round(2).to_string())
ax = fam_w.drop(columns="% de features").T.plot(kind="bar", stacked=True, figsize=(10, 3.8),
                                                 color=[FAM_COLOR[f] for f in fam_w.index])
ax.set(ylabel="suma de loadings²", title="¿De qué familia es cada componente?")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left"); plt.tight_layout(); plt.show()

In [ ]:
# LDA: 2 direcciones que mejor separan las 3 clases (priors iguales, igual que en el pipeline)
lda_viz = LinearDiscriminantAnalysis(n_components=2, solver="eigen", shrinkage="auto",
                                     priors=np.full(len(CLASSES), 1 / len(CLASSES))).fit(Ztr, ytr)
Ltr, Lva = lda_viz.transform(Ztr), lda_viz.transform(Zva)
print("separación entre clases explicada por LD1, LD2:", lda_viz.explained_variance_ratio_.round(3))

fig, axs = plt.subplots(1, 2, figsize=(13, 5.5), sharex=True, sharey=True)
for ax, L, yy, title in [(axs[0], Ltr, ytr, "train (con el que se ajusta: optimista)"),
                         (axs[1], Lva, yva, "val (no visto: honesto)")]:
    for j, c in enumerate(CLASSES):
        m = yy == j
        ax.scatter(L[m, 0], L[m, 1], label=f"{c} ({m.sum()})", **style(c))
    ax.set(xlabel="LD1", ylabel="LD2", title=f"LDA: {title}")
    ax.legend(markerscale=3)
plt.tight_layout(); plt.show()

# Coeficientes de cada discriminante sobre las features estandarizadas. Ojo: con features muy correlacionadas los
# coeficientes son inestables (se reparten el peso entre features parecidas); úsalo como orientación, no como verdad.
coef = pd.DataFrame(lda_viz.scalings_[:, :2], index=FEATS, columns=["LD1", "LD2"])
fig, axs = plt.subplots(1, 2, figsize=(11, 5.5))
for ax, ld in zip(axs, coef.columns):
    top = coef[ld].abs().sort_values(ascending=False).head(15).index[::-1]
    ax.barh(top, coef.loc[top, ld], color=[FAM_COLOR[family(c)] for c in top])
    ax.axvline(0, color="k", lw=0.6)
    ax.set_title(f"{ld}: top-15 coeficientes"); ax.tick_params(axis="y", labelsize=7)
plt.tight_layout(); plt.show()

## 2. Modelos y validación cruzada (todas las reducciones)

- **todas / top-K:** imputación → [top-K] → cuantil (todos los modelos, `SCALE_ALL`) → RUS+SMOTE → modelo.
- **PCA / LDA:** imputación → cuantil → `StandardScaler` → PCA / LDA → RUS+SMOTE → modelo (no se vuelve a escalar).
- La parte de reducción (imputación + top-K / PCA / LDA) se ajusta **una vez por fold**, solo con el train del fold y
  **sin remuestrear**, y la comparten todos los modelos. Después todos los pares **modelo × fold** se entrenan a la vez.
- Las métricas de train (para ver el overfitting) se calculan sobre el train del fold **original**, sin remuestrear.
- `Dummy` es el suelo (kappa 0, balanced accuracy 0.33). Para el SVM, el AUC se calcula con un softmax de `decision_function`.

In [ ]:
MAIN = "kappa"                     # métrica para ordenar y elegir el mejor modelo
SCORING = ["kappa", "bal_acc", "f1_macro", "auc_ovr", "ap_macro"] + [f"recall_{c}" for c in CLASSES]

# sin class_weight / sample_weight: el desbalance ya lo corrige el remuestreo
MODELS = {   # nombre: (estimador, ¿necesita escalado?, ¿usa sample_weight balanceado?)
    "Dummy": (DummyClassifier(strategy="prior"), False, False),
    "LogReg (baseline)": (LogisticRegression(C=0.5, max_iter=5000), True, False),
    "LDA": (LinearDiscriminantAnalysis(solver="eigen", shrinkage="auto"), True, False),
    "KNN": (KNeighborsClassifier(n_neighbors=15, weights="distance"), True, False),
    "SVM-RBF": (SVC(C=1.0, gamma="scale"), True, False),
    "Decision Tree": (DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, random_state=SEED), False, False),
    "Random Forest": (RandomForestClassifier(n_estimators=400, min_samples_leaf=2, random_state=SEED, n_jobs=1), False, False),
    "HistGradientBoosting": (HistGradientBoostingClassifier(learning_rate=0.05, max_iter=400, max_depth=4,
                                                            early_stopping=False,
                                                            random_state=SEED), False, False),
}
if XGBClassifier is not None:
    MODELS["XGBoost"] = (XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=4, min_child_weight=3,
                                       subsample=0.8, colsample_bytree=0.5, reg_lambda=1.0, tree_method="hist",
                                       objective="multi:softprob", eval_metric="mlogloss", n_jobs=1,
                                       random_state=SEED), False, False)

slug = lambda s: "".join(ch if ch.isalnum() else "_" for ch in s).strip("_").lower()


def make_selector(k, n_jobs=1):
    """Top-k features según la importancia de un Random Forest (con pesos balanceados).
    Igual que en los otros notebooks: se ajusta con el train del fold sin remuestrear."""
    est = RandomForestClassifier(n_estimators=300, max_features="sqrt", min_samples_leaf=3,
                                 class_weight="balanced_subsample", random_state=SEED, n_jobs=n_jobs)
    return SelectFromModel(est, max_features=k, threshold=-np.inf)


def make_scaler():
    """Cuantil (-> normal) + StandardScaler, antes de PCA / LDA.
    El cuantil quita colas y outliers; el StandardScaler garantiza media 0 y desviación 1 exactas en todas las features
    (con features con muchos valores repetidos el cuantil solo no lo asegura). Sin esto, PCA se lo llevarían las
    features con más varianza por su escala, no por su información."""
    return [("scale", QuantileTransformer(n_quantiles=200, output_distribution="normal", random_state=SEED)),
            ("std", StandardScaler())]


def make_reducer(red):
    kind, n = red
    if kind == "pca":
        return PCA(n_components=n, random_state=SEED)
    if kind == "lda":      # priors iguales: que la proyección no la dicte el desbalance (se ajusta antes del remuestreo)
        return LinearDiscriminantAnalysis(n_components=n, solver="eigen", shrinkage="auto",
                                          priors=np.full(len(CLASSES), 1 / len(CLASSES)))
    raise ValueError(f"reducción desconocida: {red}")


is_projected = lambda red: red is not None and red[0] in ("pca", "lda")


def make_prep(red, n_jobs=1):
    """Parte que se ajusta UNA vez por fold (solo con el train del fold, sin remuestrear) y comparten todos los modelos:
       None           -> imputación
       ("topk", K)    -> imputación -> top-K (Random Forest)
       ("pca"/"lda")  -> imputación -> cuantil -> StandardScaler -> PCA / LDA"""
    steps = [("imp", SimpleImputer(strategy="median"))]
    if red is not None and red[0] == "topk":
        steps.append(("sel", make_selector(red[1], n_jobs=n_jobs)))
    elif is_projected(red):
        steps += make_scaler() + [("red", make_reducer(red))]
    elif red is not None:
        raise ValueError(f"reducción desconocida: {red}")
    return Pipeline(steps)


def make_head(model, scale, projected=False):
    """[escalado] -> sampler -> modelo. ImbPipeline remuestrea solo en fit(); predict() no lo toca.
    Con PCA / LDA los datos ya llegan escalados y proyectados: no se vuelven a escalar."""
    steps = []
    if (scale or SCALE_ALL) and not projected:
        steps.append(("scale", QuantileTransformer(n_quantiles=200, output_distribution="normal", random_state=SEED)))
    return ImbPipeline(steps + [("sampler", clone(SAMPLER)), ("clf", clone(model))])


def fit_head(model, scale, sw, X, y, projected=False):
    warnings.filterwarnings("ignore")
    kw = {"clf__sample_weight": compute_sample_weight("balanced", y)} if sw else {}
    return make_head(model, scale, projected).fit(X, y, **kw)


def predict_scores(est, X):
    """Probabilidades (n, 3). Si el modelo no tiene predict_proba (SVM), softmax de decision_function."""
    if hasattr(est, "predict_proba"):
        return est.predict_proba(X)
    d = est.decision_function(X)
    e = np.exp(d - d.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)


def mc_metrics(y, P, pred):
    rec = recall_score(y, pred, labels=list(range(len(CLASSES))), average=None, zero_division=0)
    out = {"kappa": cohen_kappa_score(y, pred), "bal_acc": balanced_accuracy_score(y, pred),
           "f1_macro": f1_score(y, pred, average="macro"),
           "auc_ovr": roc_auc_score(y, P, multi_class="ovr", average="macro", labels=list(range(len(CLASSES))))}
    Y = label_binarize(y, classes=list(range(len(CLASSES))))         # one-vs-rest
    ap = [average_precision_score(Y[:, j], P[:, j]) for j in range(len(CLASSES))]   # área bajo la curva PR
    out["ap_macro"] = float(np.mean(ap))
    out.update({f"recall_{c}": r for c, r in zip(CLASSES, rec)})
    out.update({f"ap_{c}": v for c, v in zip(CLASSES, ap)})
    return out


def _prep_fold(X, y, a, b, red, n_jobs=1):
    """Imputación (+ top-K, o escalado + PCA / LDA) ajustada solo con el train del fold."""
    warnings.filterwarnings("ignore")
    p = make_prep(red, n_jobs=n_jobs).fit(X[a], y[a])
    return p.transform(X[a]), p.transform(X[b])


def _eval_fold(model, scale, sw, X, y, a, b, prepped=None, projected=False):
    warnings.filterwarnings("ignore")
    if prepped is None:                       # sin reducción la imputación es instantánea: se hace aquí mismo
        prepped = _prep_fold(X, y, a, b, None)
    Xa, Xb = prepped
    m = fit_head(model, scale, sw, Xa, y[a], projected)
    test = mc_metrics(y[b], predict_scores(m, Xb), m.predict(Xb))      # fold de validación (no visto)
    train = mc_metrics(y[a], predict_scores(m, Xa), m.predict(Xa))     # mismo train del fold (overfitting)
    return test | {f"train_{key}": v for key, v in train.items()}


def run_cv_many(subsets, red, models=MODELS, return_raw=False):
    """CV de varios subconjuntos de features x varios modelos, todo en paralelo, con la reducción `red`
    (una de las de REDUCTIONS). subsets: {nombre: X}. Devuelve una tabla indexada por (subconjunto, modelo)."""
    y = ytr
    folds = list(skf.split(np.zeros(len(y)), y))
    Xs = {n: np.asarray(X, dtype=np.float64) for n, X in subsets.items()}
    prepped = {}
    if red is not None:                                    # 1) reducción una vez por (subconjunto, fold)
        keys = [(n, f) for n in Xs for f in range(len(folds))]
        inner = max(1, N_JOBS // len(keys))                # cores sobrantes -> árboles del RF selector (top-K)
        out = Parallel(n_jobs=min(N_JOBS, len(keys)))(
            delayed(_prep_fold)(Xs[n], y, *folds[f], red, inner) for n, f in keys)
        prepped = dict(zip(keys, out))
    jobs = [(n, m, f) for n in Xs for m in models for f in range(len(folds))]   # 2) todos los modelo x fold a la vez
    res = Parallel(n_jobs=N_JOBS)(
        delayed(_eval_fold)(*models[m], Xs[n] if red is None else None, y, *folds[f], prepped.get((n, f)),
                            is_projected(red))
        for n, m, f in jobs)
    df = pd.DataFrame(res).assign(subset=[j[0] for j in jobs], model=[j[1] for j in jobs])
    g = df.groupby(["subset", "model"], sort=False)[SCORING]
    mean, std = g.mean(), g.std(ddof=0)
    table = pd.DataFrame({m: mean[m].map("{:.3f}".format) + " ± " + std[m].map("{:.3f}".format) for m in SCORING})
    table["_main"] = mean[MAIN]
    return (table, df) if return_raw else table


def run_cv(X, red, models=MODELS, return_raw=False):
    table, raw = run_cv_many({"X": X}, red, models, return_raw=True)
    table = table.droplevel("subset").sort_values("_main", ascending=False)
    return (table, raw) if return_raw else table


print(f"CV {CV_FOLDS} folds en train ({len(Xtr)} imágenes), {len(MODELS)} modelos x {len(REDUCTIONS)} reducciones, "
      f"remuestreo {RESAMPLING}, N_JOBS = {N_JOBS}\n")
cv_tables, cv_raw = {}, {}
for rname, red in REDUCTIONS.items():
    t, raw = run_cv(Xtr, red, return_raw=True)
    cv_tables[rname], cv_raw[rname] = t, raw
    t.drop(columns="_main").to_csv(OUT_DIR / f"cv_{slug(rname)}.csv")
    print(f"--- {rname}  {red if red is not None else f'(las {len(FEATS)} features)'} ---")
    print(t.drop(columns="_main").to_string(), "\n")

### Resumen: modelo × reducción
Media de los 5 folds de cada modelo con cada reducción. Lo normal es que PCA ayude (o no perjudique) a los modelos
lineales y de distancias (LogReg, LDA, SVM, KNN) y perjudique a los árboles. Mira el kappa **y** el recall de SCC.

In [ ]:
mean_of = lambda metric: pd.DataFrame({r: raw.groupby("model", sort=False)[metric].mean()
                                       for r, raw in cv_raw.items()}).reindex(list(MODELS))
summ = {m: mean_of(m) for m in ["kappa", "bal_acc", "recall_scc"]}
for m, t in summ.items():
    print(f"--- CV {m} (media de {CV_FOLDS} folds) ---")
    print(t.round(3).to_string(), "\n")
    t.round(4).to_csv(OUT_DIR / f"summary_cv_{m}.csv")

fig, axs = plt.subplots(len(summ), 1, figsize=(1.0 * len(REDUCTIONS) + 4, len(summ) * (0.45 * len(MODELS) + 1.2)))
for ax, (m, t) in zip(np.atleast_1d(axs), summ.items()):
    t = t.drop(index="Dummy", errors="ignore")
    v = t.to_numpy(dtype=float)
    im = ax.imshow(v, cmap="viridis", aspect="auto")
    ax.set_xticks(range(t.shape[1]), t.columns, rotation=30, ha="right")
    ax.set_yticks(range(t.shape[0]), t.index)
    for (i, j), x in np.ndenumerate(v):
        ax.text(j, i, f"{x:.2f}", ha="center", va="center", fontsize=8, color="w" if x < np.nanmean(v) else "k")
    ax.set_title(f"CV {m}")
    fig.colorbar(im, ax=ax, fraction=0.025)
plt.tight_layout(); plt.show()

best_red = max(cv_tables, key=lambda r: cv_tables[r]["_main"].iloc[0])     # variante cuyo mejor modelo tiene más MAIN
print(f"Mejor combinación en CV ({MAIN}): {best_red} + {cv_tables[best_red].index[0]} "
      f"= {cv_tables[best_red]['_main'].iloc[0]:.3f}")

### ¿Hay overfitting? Train vs CV
Para `todas` y para la mejor reducción de la CV. Rendimiento sobre el mismo train con el que se entrena cada fold frente
al fold de validación (sale de la CV de arriba, no se entrena nada nuevo). En Random Forest y XGBoost es normal que train
esté cerca de 1.0; en KNN con `weights="distance"` train es siempre 1.0 (cada imagen es su propio vecino). Lo que manda es la CV.

In [ ]:
def train_vs_cv(raw):
    g = raw.groupby("model", sort=False)
    t = pd.DataFrame({"train_kappa": g["train_kappa"].mean(), "cv_kappa": g["kappa"].mean(),
                      "train_bal_acc": g["train_bal_acc"].mean(), "cv_bal_acc": g["bal_acc"].mean(),
                      "train_recall_scc": g["train_recall_scc"].mean(), "cv_recall_scc": g["recall_scc"].mean()})
    t.insert(2, "gap_kappa", t["train_kappa"] - t["cv_kappa"])
    t.insert(5, "gap_bal_acc", t["train_bal_acc"] - t["cv_bal_acc"])
    return t.sort_values("cv_kappa", ascending=False).round(3)


for r in dict.fromkeys(x for x in ("todas", best_red) if x in cv_raw):
    ovf = train_vs_cv(cv_raw[r])
    print(f"--- {r}: train vs CV ---")
    print(ovf.to_string(), "\n")
    ovf.to_csv(OUT_DIR / f"train_vs_cv_{slug(r)}.csv")

## 3. Ablación por familias
El mejor modelo de la CV (por kappa) con cada familia sola y combinadas (todas en paralelo). Ojo con las de **imagen entera** (`gcolor`, `gtex`):
pueden captar la cámara o el fondo en vez de la lesión.

In [ ]:
ABL_MODEL = cv_tables.get("todas", cv_tables[best_red]).index[0]   # el mejor de la CV con todas las features
fam_cols = {f: [c for c in FEATS if family(c) == f] for f in ["shape", "color", "tex", "gcolor", "gtex"]}
lesion = fam_cols["shape"] + fam_cols["color"] + fam_cols["tex"]
imagen = fam_cols["gcolor"] + fam_cols["gtex"]
subsets = {"solo forma": fam_cols["shape"], "solo color": fam_cols["color"], "solo textura": fam_cols["tex"],
           "solo gcolor": fam_cols["gcolor"], "solo gtex": fam_cols["gtex"],
           "forma + color": fam_cols["shape"] + fam_cols["color"], "color + textura": fam_cols["color"] + fam_cols["tex"],
           "lesión (shape+color+tex)": lesion, "imagen entera (gcolor+gtex)": imagen, "todas": FEATS}
abl = run_cv_many({name: Xtr[cols] for name, cols in subsets.items()}, red=None,
                  models={ABL_MODEL: MODELS[ABL_MODEL]}).droplevel("model")
abl.insert(0, "n_features", [len(c) for c in subsets.values()])
print(f"Ablación con {ABL_MODEL}\n")
print(abl.drop(columns="_main").to_string())
abl.drop(columns="_main").to_csv(OUT_DIR / "ablation_families.csv")

## 4. Evaluación final en val
Se usa la reducción (todas, top-K, PCA o LDA) cuyo mejor modelo haya dado mejor kappa en la CV, **sin mirar val**
(se puede fijar a mano con `FINAL_RED_NAME`, pero decidiéndolo con la CV). Cada modelo se entrena con **todo train**
(en paralelo) y se evalúa una sola vez en val. El IC95 % del kappa sale de un bootstrap sobre val.
El sampler solo sirve para entrenar, así que el pipeline final se guarda **sin** él
(imputación → [top-K | cuantil → StandardScaler → PCA/LDA] → [escalado] → modelo). Predice exactamente igual.

In [ ]:
FINAL_RED_NAME = best_red          # elegida por la CV; se puede fijar a mano (p. ej. "PCA-30"), pero con la CV, no con val
FINAL_RED = REDUCTIONS[FINAL_RED_NAME]
res_cv = cv_tables[FINAL_RED_NAME]
best = res_cv.index[0]                                  # mejor modelo según la CV (no según val)
print(f"Reducción para el modelo final: {FINAL_RED_NAME} {FINAL_RED if FINAL_RED is not None else ''} | mejor en CV: {best}\n")


def boot_ci(y, pred, fn=cohen_kappa_score, n=1000, seed=0):
    rng = np.random.RandomState(seed)
    vals = [fn(y[i], pred[i]) for i in (rng.choice(len(y), len(y), replace=True) for _ in range(n))]
    return np.percentile(vals, [2.5, 97.5])


proj = is_projected(FINAL_RED)
prep = make_prep(FINAL_RED, n_jobs=N_JOBS).fit(Xtr, ytr)          # imputación (+ reducción) una sola vez
Xtr_s, Xva_s = prep.transform(Xtr), prep.transform(Xva)
print(f"dimensiones que ve el modelo: {Xtr_s.shape[1]}\n")
heads = Parallel(n_jobs=N_JOBS)(delayed(fit_head)(est, scale, sw, Xtr_s, ytr, proj) for est, scale, sw in MODELS.values())

fitted, rows, probas, preds = {}, [], {}, {}
for name, head in zip(MODELS, heads):
    pipe = Pipeline(prep.steps + [s for s in head.steps if s[0] != "sampler"])   # ya entrenado, sin el sampler (acepta las features crudas)
    P, pred = predict_scores(pipe, Xva), pipe.predict(Xva)
    lo, hi = boot_ci(yva, pred)
    fitted[name], probas[name], preds[name] = pipe, P, pred
    rows.append({"modelo": name, **mc_metrics(yva, P, pred), "kappa IC95%": f"[{lo:.3f}, {hi:.3f}]"})
val_res = pd.DataFrame(rows).set_index("modelo").sort_values("kappa", ascending=False)
print(f"Val ({len(Xva)} imágenes):\n")
print(val_res.round(3).to_string())
val_res.round(4).to_csv(OUT_DIR / "val_results.csv")

### Informativo: el mejor modelo de cada reducción, en val
Para cada variante de `REDUCTIONS`, su mejor modelo de la CV entrenado con todo train y evaluado en val. Sirve para ver si
val **confirma** la CV, **no para elegir**: si eliges mirando esta tabla, val deja de ser una estimación honesta.
Con 1270 imágenes (94 SCC), diferencias dentro del IC95 % son ruido. Si val contradice a la CV claramente (fuera del IC),
es una pista de que algo pasa (por ejemplo, que train y val no se parecen).

In [ ]:
VAL_ALL_REDUCTIONS = True        # False para saltarlo (entrena un modelo más por reducción)


def _best_of_reduction(rname):
    warnings.filterwarnings("ignore")
    red, m = REDUCTIONS[rname], cv_tables[rname].index[0]
    p = make_prep(red).fit(Xtr, ytr)
    h = fit_head(*MODELS[m], p.transform(Xtr), ytr, is_projected(red))
    Xv = p.transform(Xva)
    return m, predict_scores(h, Xv), h.predict(Xv)


if VAL_ALL_REDUCTIONS:
    out = Parallel(n_jobs=min(N_JOBS, len(REDUCTIONS)))(delayed(_best_of_reduction)(r) for r in REDUCTIONS)
    rows = []
    for rname, (m, P, pred) in zip(REDUCTIONS, out):
        lo, hi = boot_ci(yva, pred)
        rows.append({"reducción": rname, "mejor modelo (CV)": m, "cv_kappa": cv_tables[rname]["_main"].iloc[0],
                     **mc_metrics(yva, P, pred), "kappa IC95%": f"[{lo:.3f}, {hi:.3f}]"})
    val_by_red = pd.DataFrame(rows).set_index("reducción")
    cols = ["mejor modelo (CV)", "cv_kappa", "kappa", "kappa IC95%", "bal_acc", "f1_macro", "auc_ovr", "ap_macro"] + \
           [f"recall_{c}" for c in CLASSES]
    print(val_by_red[cols].round(3).to_string())
    val_by_red.round(4).to_csv(OUT_DIR / "val_best_per_reduction.csv")

### Matrices de confusión de todos los modelos (val, normalizadas por fila = recall de cada clase)

In [ ]:
names = list(fitted)
nc = 4
nr = int(np.ceil(len(names) / nc))
fig, axs = plt.subplots(nr, nc, figsize=(4.2 * nc, 3.9 * nr), squeeze=False)
for ax, name in zip(axs.ravel(), names):
    ConfusionMatrixDisplay.from_predictions(yva, preds[name], labels=list(range(len(CLASSES))), display_labels=CLASSES,
                                            normalize="true", values_format=".2f", ax=ax, colorbar=False)
    ax.set_title(f"{name}\nkappa {val_res.loc[name, 'kappa']:.3f} | bal_acc {val_res.loc[name, 'bal_acc']:.3f}", fontsize=10)
for ax in axs.ravel()[len(names):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

### Mejor modelo (elegido por CV): detalle

In [ ]:
print(f"Mejor modelo en CV: {best}\n")
print(classification_report(yva, preds[best], target_names=CLASSES, digits=3))

fig, axs = plt.subplots(1, 2, figsize=(11, 4.3))
for ax, norm, fmt, title in [(axs[0], None, "d", "conteos"), (axs[1], "true", ".2f", "normalizada por fila")]:
    ConfusionMatrixDisplay.from_predictions(yva, preds[best], labels=list(range(len(CLASSES))), display_labels=CLASSES,
                                            normalize=norm, values_format=fmt, ax=ax, colorbar=False)
    ax.set_title(f"{best}: {title}")
plt.tight_layout(); plt.show()

### Curvas precision-recall (val)
Con clases desbalanceadas, la curva PR es más informativa que la ROC para la clase minoritaria: la ROC usa la tasa de
falsos positivos, que casi no se mueve porque SCC tiene muchísimos negativos (MEL + BCC); la PR usa la **precisión**,
que sí cae con cada falso positivo. El nivel del azar en PR es la **prevalencia** de la clase (línea discontinua),
no 0.5 como en la ROC.

- Izquierda: el mejor modelo, una curva por clase (one-vs-rest).
- Derecha: SCC para todos los modelos.
- El punto marca dónde cae la predicción actual (argmax), es decir, el recall y la precisión que se obtienen hoy.

In [ ]:
Yva = label_binarize(yva, classes=list(range(len(CLASSES))))
colors = {"mel": "#1f77b4", "bcc": "#ff7f0e", "scc": "#2ca02c"}
fig, axs = plt.subplots(1, 2, figsize=(13, 5))

for j, c in enumerate(CLASSES):
    prec, rec, _ = precision_recall_curve(Yva[:, j], probas[best][:, j])
    ap = average_precision_score(Yva[:, j], probas[best][:, j])
    axs[0].plot(rec, prec, color=colors.get(c), label=f"{c} (AP {ap:.3f})")
    axs[0].axhline(Yva[:, j].mean(), color=colors.get(c), ls="--", lw=0.8)
    hit = preds[best] == j
    if hit.any():
        axs[0].plot(recall_score(Yva[:, j], hit), (Yva[hit, j]).mean(), "o", color=colors.get(c))
axs[0].set(xlabel="recall", ylabel="precisión", title=f"{best}: PR por clase", xlim=(0, 1), ylim=(0, 1.02))
axs[0].legend(loc="lower left")

j = CLASSES.index("scc")
for name in fitted:
    if name == "Dummy":
        continue
    prec, rec, _ = precision_recall_curve(Yva[:, j], probas[name][:, j])
    ap = average_precision_score(Yva[:, j], probas[name][:, j])
    axs[1].plot(rec, prec, lw=2 if name == best else 1, label=f"{name} (AP {ap:.3f})")
axs[1].axhline(Yva[:, j].mean(), color="k", ls="--", lw=0.8, label=f"azar = prevalencia ({Yva[:, j].mean():.3f})")
axs[1].set(xlabel="recall", ylabel="precisión", title="SCC: todos los modelos", xlim=(0, 1), ylim=(0, 1.02))
axs[1].legend(fontsize=8, loc="upper right")
plt.tight_layout(); plt.show()

### Guardar los modelos
Se guarda cada modelo final (entrenado con todo train) en `ml_results_multiclass_pca_lda/models/` como un `.joblib` con el **pipeline completo**
(imputación → [selección | escalado → PCA/LDA] → [escalado] → modelo), más una copia del mejor como `best_model.joblib` y un `models_info.json`
con la lista de features (en orden), las etiquetas, el remuestreo usado, las métricas en val y las versiones de las
librerías. Los pipelines guardados no incluyen el sampler.
(`*.joblib` está en el `.gitignore`: no se suben a GitHub.)

Para usarlos después (por ejemplo con las features del test), hacen falta **las mismas versiones** de sklearn / xgboost:
```python
import json, joblib, pandas as pd
d = ROOT / "ml" / "ml_results_multiclass_pca_lda" / "models"
info = json.load(open(d / "models_info.json", encoding="utf-8"))
model = joblib.load(d / "best_model.joblib")
X_test = pd.read_csv("features_test.csv")[info["features"]]     # mismas columnas, mismo orden
pred = model.predict(X_test)                                    # 0 = bcc, 1 = mel, 2 = scc (orden de CLASSES)
```

In [ ]:
TASK = "multiclass"
LABELS = dict(enumerate(CLASSES))
SELECTION = f"{FINAL_RED_NAME} {FINAL_RED}" if FINAL_RED is not None else "todas las features"

import json
import platform
from datetime import datetime

import joblib
import sklearn

MODEL_DIR = OUT_DIR / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
slug = lambda s: "".join(ch if ch.isalnum() else "_" for ch in s).strip("_").lower()

# cada archivo es el pipeline COMPLETO ya entrenado con todo train:
# imputación -> [selección top-K] -> [escalado] -> modelo. Recibe las features crudas (FEATS, en ese orden).
saved = {}
for name, pipe in fitted.items():
    if name == "Dummy":
        continue
    path = MODEL_DIR / f"{slug(name)}.joblib"
    joblib.dump(pipe, path, compress=3)
    saved[name] = path.name
joblib.dump(fitted[best], MODEL_DIR / "best_model.joblib", compress=3)

# comprobación: el modelo recargado predice exactamente lo mismo en val
assert (joblib.load(MODEL_DIR / "best_model.joblib").predict(Xva) == preds[best]).all()

try:
    import xgboost
    xgb_version = xgboost.__version__
except Exception:
    xgb_version = None

info = {
    "task": TASK,
    "created": datetime.now().isoformat(timespec="seconds"),
    "best_model": best,
    "files": {**saved, "best": "best_model.joblib"},
    "labels": LABELS,
    "feature_selection": SELECTION,
    "resampling": {"method": "RandomUnderSampler + SMOTE", "target": TARGET, "smote_k": SMOTE_K},   # solo se usó al entrenar
    "n_features_in": len(FEATS),
    "features": FEATS,                                   # columnas de entrada, en este orden
    "train_csv": str(TRAIN_CSV),
    "val_metrics": val_res.reset_index().to_dict(orient="records"),
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__, "xgboost": xgb_version,
                 "numpy": np.__version__, "pandas": pd.__version__},
}
(MODEL_DIR / "models_info.json").write_text(json.dumps(info, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print(f"Modelos guardados en {MODEL_DIR}  (mejor: {best} -> best_model.joblib)\n")
for f in sorted(MODEL_DIR.glob("*")):
    print(f"  {f.name:<35s} {f.stat().st_size / 1e6:7.1f} MB")

### ¿Qué features usa el mejor modelo?
Importancia por permutación en val (caída del **kappa** al barajar cada feature). Con top-K solo se permutan las K
seleccionadas. Con PCA / LDA se permutan las **features originales** (cada una afecta a todos los componentes), así que
la importancia sigue siendo interpretable aunque el modelo vea componentes.

In [ ]:
pipe = fitted[best]
if "sel" in pipe.named_steps:
    sel = pipe.named_steps["sel"].get_support()
    X_pi, est_pi, names_pi = pipe.named_steps["imp"].transform(Xva)[:, sel], pipe[2:], np.array(FEATS)[sel]
else:
    X_pi, est_pi, names_pi = pipe.named_steps["imp"].transform(Xva), pipe[1:], np.array(FEATS)
pi = permutation_importance(est_pi, X_pi, yva, scoring=make_scorer(cohen_kappa_score), n_repeats=5,
                            random_state=SEED, n_jobs=N_JOBS)
pi = pd.Series(pi.importances_mean, index=names_pi).sort_values(ascending=False)
pi.rename("importancia").to_csv(OUT_DIR / "permutation_importance_best.csv")
top = pi.head(20)[::-1]
plt.figure(figsize=(8, 6))
plt.barh(top.index, top.values, color=[FAM_COLOR[family(c)] for c in top.index])
plt.title(f"Importancia por permutación en val ({best})")
plt.xlabel("caída del kappa"); plt.tight_layout(); plt.show()
print("Importancia total por familia:", pi.groupby(lambda c: family(c)).sum().round(3).to_dict())

## 5. Control de sesgo: ¿se puede adivinar la clase solo con la resolución de la imagen?
Si la resolución original (que delata el dataset de origen: HAM10000, BCN20000, MSK...) ya predice bien la clase, parte
del rendimiento puede venir de la fuente y no de la lesión. Usa `processed/manifest.csv` (`orig_h`, `orig_w`).

In [ ]:
man_path = ROOT / "processed" / "manifest.csv"
if not man_path.exists():
    print(f"No encuentro {man_path}: se omite este control.")
else:
    man = pd.read_csv(man_path).drop_duplicates("name").set_index("name")
    cols = ["orig_h", "orig_w"]
    m_tr, m_va = man.reindex(tr["name"])[cols].to_numpy(), man.reindex(va["name"])[cols].to_numpy()
    g = HistGradientBoostingClassifier(class_weight="balanced", random_state=SEED).fit(m_tr, ytr)
    p = g.predict(m_va)
    print(f"Solo con la resolución original -> val kappa {cohen_kappa_score(yva, p):.3f}, "
          f"bal_acc {balanced_accuracy_score(yva, p):.3f}\n")
    res_str = man.reindex(va["name"])["orig_w"].astype("Int64").astype(str) + "x" + \
              man.reindex(va["name"])["orig_h"].astype("Int64").astype(str)
    ct = pd.crosstab(res_str.to_numpy(), va["subtype"].to_numpy())
    print(ct.reindex(columns=CLASSES).loc[ct.sum(axis=1).sort_values(ascending=False).index].head(10).to_string())

    rows = []
    for r, idx in pd.Series(range(len(va)), index=res_str.to_numpy()).groupby(level=0):
        idx = idx.to_numpy()
        if len(idx) >= 30 and len(np.unique(yva[idx])) > 1:
            rows.append((r, len(idx), cohen_kappa_score(yva[idx], preds[best][idx]),
                         balanced_accuracy_score(yva[idx], preds[best][idx])))
    print(f"\n{best} por resolución (grupos con >= 30 imágenes):")
    print(pd.DataFrame(rows, columns=["res", "n", "kappa", "bal_acc"]).round(3).to_string(index=False))

## Notas y siguientes pasos
*(Rellenar después de ejecutar.)* Comparar con v5 (mismo remuestreo, sin PCA/LDA) y el resto
(`ml/ml_results_multiclass*/val_results.csv`).

- ¿Qué reducción gana en CV? ¿Con qué modelos ayuda PCA y con cuáles empeora?
- Nº de componentes de PCA: ¿mejora al subir de 10 → 30 → 60, o se estanca? (Si se estanca pronto, sobran features.)
- ¿Qué separa LDA en el scatter de val? (¿SCC queda entre MEL y BCC o solapado con alguna?)
- Recall de SCC con la mejor reducción frente a v5.